In [ ]:
from nba_api.stats.endpoints import PlayerGameLog
import pandas as pd

player_id = 2544  # Luka
season = "2025-26"

# fetch all regular season games
gamelog = PlayerGameLog(
    player_id=player_id, season=season, season_type_all_star="Regular Season"
)

# get DataFrame
df = gamelog.get_data_frames()[0]

# sort by date descending and pick last N games
N = 20
df_last_n = df.sort_values("GAME_DATE", ascending=False).head(N)
print(df_last_n[["GAME_DATE", "MATCHUP", "PTS", "AST", "REB"]])


       GAME_DATE      MATCHUP  PTS  AST  REB
18  Nov 28, 2025  LAL vs. DAL   13    7    5
19  Nov 25, 2025  LAL vs. LAC   25    6    6
20  Nov 23, 2025    LAL @ UTA   17    8    6
21  Nov 18, 2025  LAL vs. UTA   11   12    3
0   Jan 15, 2026  LAL vs. CHA   29    6    9
1   Jan 13, 2026  LAL vs. ATL   31   10    9
2   Jan 12, 2026    LAL @ SAC   22    3    4
3   Jan 09, 2026  LAL vs. MIL   26   10    9
4   Jan 06, 2026    LAL @ NOP   30    8    8
5   Jan 04, 2026  LAL vs. MEM   26   10    7
6   Jan 02, 2026  LAL vs. MEM   31    6    9
7   Dec 30, 2025  LAL vs. DET   17    4    4
8   Dec 28, 2025  LAL vs. SAC   24    5    3
9   Dec 25, 2025  LAL vs. HOU   18    5    2
10  Dec 23, 2025    LAL @ PHX   23    6    2
11  Dec 20, 2025    LAL @ LAC   36    3    4
12  Dec 18, 2025    LAL @ UTA   28   10    7
13  Dec 14, 2025    LAL @ PHX   26    4    3
14  Dec 10, 2025  LAL vs. SAS   19    8   15
15  Dec 07, 2025    LAL @ PHI   29    6    7


: 

In [5]:
# test_schedule.py
from nba_api.stats.endpoints import scheduleleaguev2

season = "2025-26"
sched = scheduleleaguev2.ScheduleLeagueV2(season=season)
df_games = sched.season_games.get_data_frame()

df_games = df_games[
    [
        "seasonYear",
        "homeTeam_teamId",
        "homeTeam_teamTricode",
        "awayTeam_teamId",
        "awayTeam_teamTricode",
        "gameSubtype",
    ]
]
# df_games.display()


In [6]:
df_games.to_csv("./testing.csv")

In [3]:
import sys
from pathlib import Path

BACKEND_DIR = Path().resolve().parents[1]
sys.path.insert(0, str(BACKEND_DIR))


import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from app.core.config import settings

# Load prediction logs with actuals
engine = create_engine(settings.DATABASE_URL.replace("+asyncpg", ""))
df = pd.read_sql(
    """
    SELECT player_id, stat_type, game_date, pred_value, pred_p10, actual_value, confidence
    FROM prediction_logs
    WHERE actual_value IS NOT NULL AND game_date IS NOT NULL
    """,
    engine,
)


def add_threshold(row):
    if row["stat_type"] == "points":
        if pd.notnull(row["pred_value"]) and pd.notnull(row["pred_p10"]):
            return (row["pred_value"] + row["pred_p10"]) / 2.0
        return np.nan
    return row["pred_p10"]


df["threshold"] = df.apply(add_threshold, axis=1)
df = df[pd.notnull(df["threshold"])]
df["under"] = df["actual_value"] < df["threshold"]
df = df.sort_values(["stat_type", "player_id", "game_date"]).reset_index(drop=True)
df["under_next"] = df.groupby(["stat_type", "player_id"])["under"].shift(-1)


def summarize(stat):
    d = df[df["stat_type"] == stat].copy()
    overall_under_rate = d["under"].mean()
    cond_rate = d.loc[d["under"], "under_next"].mean()

    # Define "good players" by recent confidence (>= 70) with at least 10 samples
    conf = (
        d.dropna(subset=["confidence"])
        .sort_values(["player_id", "game_date"], ascending=[True, False])
        .groupby("player_id")
        .head(20)
    )
    good_players = conf.groupby("player_id")["confidence"].mean().reset_index()
    good_players = good_players[good_players["confidence"] >= 70]

    good_ids = set(good_players["player_id"])
    d_good = d[d["player_id"].isin(good_ids)]
    good_overall = d_good["under"].mean()
    good_cond = d_good.loc[d_good["under"], "under_next"].mean()

    return {
        "stat": stat,
        "overall_under_rate": overall_under_rate,
        "under_next_given_under": cond_rate,
        "good_overall_under_rate": good_overall,
        "good_under_next_given_under": good_cond,
    }


summary = pd.DataFrame(
    [summarize(s) for s in ["points", "assists", "rebounds", "threept"]]
)
summary


ModuleNotFoundError: No module named 'app'

In [4]:
from pathlib import Path

print(Path.cwd())


c:\Users\kalz9\OneDrive\Documents\onedrive\Documents\Gamblr\gamblr\backend\app
